## Imports og tilkobling

In [1]:
import sys
import os
import pandas as pd
import io
from dotenv import load_dotenv
from azure.storage.blob import BlobServiceClient

sys.path.append(os.path.abspath("../../../"))
from src.feature_engineering.forbruksdata import create_consumption_features

load_dotenv()

connection_string = os.getenv("AZURE_STORAGE_CONNECTION_STRING")

blob_service_client = BlobServiceClient.from_connection_string(connection_string)
container_name = "trafoforbruksdata"

## Hente CSV fra blob

In [ ]:
blob_name = "FRIKSTAD.csv"

blob_client = blob_service_client.get_blob_client(
    container=container_name,
    blob=blob_name
)

data = blob_client.download_blob().readall()

df = pd.read_csv(io.BytesIO(data))

## Feature engineering

In [3]:
# feature engineering
df = create_consumption_features(df)

# se resultat
df.head()

,MeteringPointAnonymous,TimestampUtc,Value,TransformerStation,ConsumptionCode,hour,weekday,month,is_weekend,is_holiday,day_night
0,480718a0-c8f0-4c4a-977d-42068f431114,2025-12-26 15:00:00,7.47,FRIKSTAD,35,15,4,12,False,False,day
1,480718a0-c8f0-4c4a-977d-42068f431114,2025-12-26 16:00:00,8.15,FRIKSTAD,35,16,4,12,False,False,day
2,480718a0-c8f0-4c4a-977d-42068f431114,2025-12-26 17:00:00,9.94,FRIKSTAD,35,17,4,12,False,False,day
3,480718a0-c8f0-4c4a-977d-42068f431114,2025-12-26 18:00:00,9.07,FRIKSTAD,35,18,4,12,False,False,day
4,480718a0-c8f0-4c4a-977d-42068f431114,2025-12-26 19:00:00,9.06,FRIKSTAD,35,19,4,12,False,False,day


## Legge til blob

In [ ]:
import io

# --- Lag CSV i minnet ---
csv_buffer = io.BytesIO()
df.to_csv(csv_buffer, index=False)
csv_buffer.seek(0)  # gå til starten av bufferet

# --- Last opp til blob ---
output_blob_name = "FRIKSTAD_processed.csv"  # kan bruke samme navn eller nytt
blob_client = blob_service_client.get_blob_client(container=container_name, blob=output_blob_name)
blob_client.upload_blob(csv_buffer, overwrite=True)  # overwrite=True overskriver eksisterende fil

print(f"{output_blob_name} lastet opp til blob!")

FRIKSTAD_processed.csv lastet opp til blob!
